# MBG Tweet Pipeline — Full Run on Colab
**From raw scraped tweets → dashboard-ready CSVs**

### Before you start:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Fill in your credentials in **Cell 3** (droplet IP, SSH key, repo URL)
3. Run all cells top-to-bottom

### Pipeline stages:
| Stage | Script | Output CSV |
|---|---|---|
| 1 | `inference.py` | `tweets_relevant.csv` |
| 2 | `tag_language.py` | `tweets_relevant_tagged.csv` |
| 3 | `preprocess_text.py` | `tweets_preprocessed.csv` |
| 4 | `run_sentiment.py` | `tweets_with_sentiment.csv` |
| 5 | `run_topics.py` | `tweets_with_topics.csv` + `topic_info.csv` |
| 6 | `validate_data_contract.py` | validation report |
| 7 | `generate_manifest.py` | `manifest.json` |
| 8 | `upload_run.py` | uploads to DO Spaces |

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Cell 1 — Mount Google Drive & Set Runtime Mode

In [14]:
from google.colab import drive
drive.mount('/content/drive')

import os
import time

# Track notebook start time for manifest duration
notebook_start_time = time.time()

# Set runtime mode — all scripts read this to switch paths and devices
os.environ["RUNTIME_MODE"] = "colab"

# Base paths on Drive
DRIVE_BASE   = "/content/drive/MyDrive/mbg"
DATA_DIR     = f"{DRIVE_BASE}/data"
RAW_DIR      = f"{DATA_DIR}/raw"
OUTPUT_DIR   = f"{DATA_DIR}/output"
LOG_DIR      = f"{DRIVE_BASE}/logs"
CODE_DIR     = "/content/mbg-pipeline"

# Create directories if they don't exist
for d in [RAW_DIR, OUTPUT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Drive mounted")
print(f"   Raw data:  {RAW_DIR}")
print(f"   Outputs:   {OUTPUT_DIR}")
print(f"   Code:      {CODE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted
   Raw data:  /content/drive/MyDrive/mbg/data/raw
   Outputs:   /content/drive/MyDrive/mbg/data/output
   Code:      /content/mbg-pipeline


## Cell 2 — Verify GPU

In [15]:
import torch

assert torch.cuda.is_available(), "❌ No GPU — go to Runtime → Change runtime type → T4 GPU"
print(f"✅ GPU ready: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✅ GPU ready: Tesla T4
   VRAM: 15.6 GB


## Cell 3 — Credentials & Codebase Setup
**Fill in your values here before running.**

In [16]:
# ─── FILL IN YOUR VALUES ────────────────────────────────────────────────────
DROPLET_IP    = "YOUR_DROPLET_IP"       # e.g. "123.456.789.0"
DROPLET_USER  = "root"                  # SSH user on your droplet
DROPLET_PORT  = "22"                    # or ngrok port if tunnelling
GITHUB_REPO   = "https://github.com/FatwaArya/mbg-analysis"  # e.g. "https://github.com/you/mbg-pipeline.git"
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys

# Option A: Clone from GitHub (recommended)
if not os.path.exists(CODE_DIR):
    print("Cloning repo...")
    subprocess.run(["git", "clone", GITHUB_REPO, CODE_DIR], check=True)
else:
    print("Repo already cloned — pulling latest...")
    subprocess.run(["git", "-C", CODE_DIR, "pull"], check=True)

# Option B: Sync from Drive (if your code is in Drive instead of GitHub)
# !rsync -av /content/drive/MyDrive/mbg-pipeline/ /content/mbg-pipeline/

# Add code directory to Python path
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f"✅ Codebase ready at {CODE_DIR}")

Repo already cloned — pulling latest...
✅ Codebase ready at /content/mbg-pipeline


## Cell 4 — Install Dependencies

In [17]:
import subprocess

req_path = f"{CODE_DIR}/requirements.txt"
assert os.path.exists(req_path), f"requirements.txt not found at {req_path}"

print("Installing dependencies (this takes ~3-5 min on first run)...")
subprocess.run(["pip", "install", "-q", "-r", req_path], check=True)

# BERTopic extras — cuML for GPU-accelerated UMAP/HDBSCAN (optional, major speedup)
try:
    subprocess.run(
        ["pip", "install", "-q", "cuml-cu11"],
        check=True, timeout=120
    )
    print("✅ cuML installed — GPU-accelerated UMAP/HDBSCAN enabled")
except Exception:
    print("⚠️  cuML not available — falling back to CPU UMAP/HDBSCAN (still fast enough)")

print("✅ All dependencies installed")

Installing dependencies (this takes ~3-5 min on first run)...
⚠️  cuML not available — falling back to CPU UMAP/HDBSCAN (still fast enough)
✅ All dependencies installed


## Cell 5 — Upload / Confirm Raw Data
Your raw scraped tweets CSV must be at `mbg/data/raw/tweets_raw.csv` on Drive before continuing.

In [18]:
import pandas as pd

RAW_CSV = f"{RAW_DIR}/final_parent_x_posts_mbg.csv"

# Option A: Already on Drive — just verify
if os.path.exists(RAW_CSV):
    df_raw = pd.read_csv(RAW_CSV)
    print(f"✅ Raw data found: {len(df_raw):,} rows")
    print(f"   Columns: {list(df_raw.columns)}")
    display(df_raw.head(3))

# Option B: Upload from local machine now
else:
    print("❌ Raw CSV not found on Drive.")
    print("   Option 1: Upload it manually to your Drive at:")
    print(f"   {RAW_CSV}")
    print()
    print("   Option 2: Upload directly in this cell:")
    print("   from google.colab import files")
    print("   uploaded = files.upload()  # select your CSV")
    print("   # then move it:")
    print("   # !mv tweets_raw.csv '{RAW_CSV}'")
    raise FileNotFoundError(f"Place tweets_raw.csv at {RAW_CSV} and re-run this cell")

✅ Raw data found: 124,144 rows
   Columns: ['id', 'text', 'created_at', 'lang', 'favorite_count', 'retweet_count', 'reply_count', 'user_id', 'user_screen_name', 'user_name', 'scrape_tab', 'chunk_since', 'chunk_until', 'search_url', 'query_raw', 'date', 'hour', 'engagement_total']


,id,text,created_at,lang,favorite_count,retweet_count,reply_count,user_id,user_screen_name,user_name,scrape_tab,chunk_since,chunk_until,search_url,query_raw,date,hour,engagement_total
0,2.043062e+18,[anak fhui bikin grup isinya lecehin perempuan...,2026-04-11 20:23:15+00:00,in,244595,67568,6091,2.040000e+18,NaN,NaN,top,2026-04-12,2026-04-13,https://x.com/search?q=%22MBG%22+(dapat+OR+ter...,"""MBG"" (dapat OR terima OR distribusi OR dikiri...",2026-04-11,20,318254
1,1.980724e+18,it’s actually insane the fact that so many ppl...,2025-10-21 19:54:03+00:00,en,164041,41183,96,2.370077e+08,NaN,NaN,top,2025-10-24,2025-10-25,https://x.com/search?q=%22MBG%22+(dapat+OR+ter...,"""MBG"" (dapat OR terima OR distribusi OR dikiri...",2025-10-21,19,205320
2,1.993732e+18,only a terrible person would complain that the...,2025-11-26 17:20:50+00:00,en,183035,17557,2869,1.140000e+18,NaN,NaN,top,2025-11-27,2025-11-28,https://x.com/search?q=(MBG+OR+%22makan+bergiz...,"(MBG OR ""makan bergizi"") (anak OR siswa OR sek...",2025-11-26,17,203461


## Cell 6 — Download Model from Spaces (if needed)
The fine-tuned IndoBERT model is required for inference. Download from Spaces if not on Drive.

In [ ]:
import subprocess

MODEL_DIR = f"{DRIVE_BASE}/model"
os.makedirs(MODEL_DIR, exist_ok=True)

# Check if model exists
if os.path.exists(f"{MODEL_DIR}/config.json"):
    print(f"✅ Model found at {MODEL_DIR}")
else:
    print("Downloading model from DigitalOcean Spaces (~475MB)...")
    print("This takes ~2-3 minutes")
    
    # Install s3cmd if not present
    !pip install -q s3cmd
    
    # Download model — credentials from Colab Secrets (click 🔑 in left sidebar)
    # Add DO_ACCESS_KEY and DO_SECRET_KEY as secrets before running
    from google.colab import userdata
    s3cfg = f'[default]\naccess_key = {userdata.get("DO_ACCESS_KEY")}\nsecret_key = {userdata.get("DO_SECRET_KEY")}\nhost_base = sgp1.digitaloceanspaces.com\nhost_bucket = %(bucket)s.sgp1.digitaloceanspaces.com\n'
    open('/root/.s3cfg', 'w').write(s3cfg)
    try:
        !s3cmd get --recursive s3://mbg-scraper-network-20260419071440/models/mbg-indobert-finetuned/ {MODEL_DIR}/
        print(f"✅ Model downloaded to {MODEL_DIR}")
    except Exception as e:
        print(f"⚠️  s3cmd download failed: {e}")
        print("Alternative: Upload model manually to Drive at:")
        print(f"   {MODEL_DIR}/")
        print("Or use Hugging Face Hub if model is published there")

## Stage 1 — Inference / Relevance Classification (GPU)
Filters raw tweets for MBG relevance. Output: `tweets_relevant.csv`

In [ ]:
import os, time
import pandas as pd
import shutil

RELEVANT_CSV = f"{OUTPUT_DIR}/tweets_relevant.csv"
PROCESSED_DIR = f"{DATA_DIR}/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

if os.path.exists(RELEVANT_CSV):
    df_check = pd.read_csv(RELEVANT_CSV)
    print(f"⏭️  Skipping — tweets_relevant.csv already exists ({len(df_check):,} rows)")
    print("   Delete the file and re-run to force re-inference")
else:
    print("Running inference.py (GPU)...")
    t0 = time.time()
    
    # FIX: inference.py is at repo root, not scripts/
    result = !RUNTIME_MODE=colab python3 {CODE_DIR}/inference.py 2>&1
    for line in result[-20:]:
        print(line)
    elapsed = time.time() - t0
    
    assert os.path.exists(RELEVANT_CSV), f"❌ Expected output not found: {RELEVANT_CSV}"
    
    # FIX: copy to processed/ for next stage (mirrors droplet script)
    shutil.copy(RELEVANT_CSV, f"{PROCESSED_DIR}/tweets_relevant.csv")
    
    df_check = pd.read_csv(RELEVANT_CSV)
    print(f"\n✅ Stage 1 done in {elapsed/60:.1f} min — {len(df_check):,} rows")
    print(f"   Relevant tweets filtered from raw corpus")

## Stage 2 — Language Tagging
Detects language of each tweet. Output: `tweets_relevant_tagged.csv`

In [ ]:
TAGGED_CSV = f"{PROCESSED_DIR}/tweets_relevant_tagged.csv"

if os.path.exists(TAGGED_CSV):
    df_check = pd.read_csv(TAGGED_CSV)
    print(f"⏭️  Skipping — tweets_relevant_tagged.csv already exists ({len(df_check):,} rows)")
else:
    print("Running tag_language.py...")
    t0 = time.time()
    
    result = !RUNTIME_MODE=colab python3 {CODE_DIR}/scripts/tag_language.py 2>&1
    for line in result[-20:]:
        print(line)
    elapsed = time.time() - t0
    
    assert os.path.exists(TAGGED_CSV), f"❌ Expected output not found: {TAGGED_CSV}"
    df_check = pd.read_csv(TAGGED_CSV)
    print(f"\n✅ Stage 2 done in {elapsed/60:.1f} min — {len(df_check):,} rows")
    print(df_check["detected_lang"].value_counts().head())

## Stage 3 — Text Preprocessing
Light + aggressive cleaning. Produces `text_clean_light` and `text_clean_topic` columns. Output: `tweets_preprocessed.csv`

In [ ]:
PREPROCESSED_CSV = f"{PROCESSED_DIR}/tweets_preprocessed.csv"

if os.path.exists(PREPROCESSED_CSV):
    df_check = pd.read_csv(PREPROCESSED_CSV)
    print(f"⏭️  Skipping — tweets_preprocessed.csv already exists ({len(df_check):,} rows)")
else:
    print("Running preprocess_text.py...")
    t0 = time.time()
    
    result = !RUNTIME_MODE=colab python3 {CODE_DIR}/scripts/preprocess_text.py 2>&1
    for line in result[-20:]:
        print(line)
    elapsed = time.time() - t0
    
    assert os.path.exists(PREPROCESSED_CSV), f"❌ Expected output not found: {PREPROCESSED_CSV}"
    df_check = pd.read_csv(PREPROCESSED_CSV)
    print(f"\n✅ Stage 3 done in {elapsed/60:.1f} min — {len(df_check):,} rows")
    print(f"   Columns: {list(df_check.columns)}")

## Stage 4 — Sentiment Analysis (GPU)
Runs dual-model sentiment on `text_clean_light`. Output: `tweets_with_sentiment.csv`

In [ ]:
SENTIMENT_CSV = f"{OUTPUT_DIR}/tweets_with_sentiment.csv"

if os.path.exists(SENTIMENT_CSV):
    df_check = pd.read_csv(SENTIMENT_CSV)
    print(f"⏭️  Skipping — tweets_with_sentiment.csv already exists ({len(df_check):,} rows)")
else:
    print("Running run_sentiment.py (GPU)...")
    t0 = time.time()
    
    result = !RUNTIME_MODE=colab python3 {CODE_DIR}/scripts/run_sentiment.py 2>&1
    for line in result[-20:]:
        print(line)
    elapsed = time.time() - t0
    
    assert os.path.exists(SENTIMENT_CSV), f"❌ Expected output not found: {SENTIMENT_CSV}"
    df_check = pd.read_csv(SENTIMENT_CSV)
    print(f"\n✅ Stage 4 done in {elapsed/60:.1f} min — {len(df_check):,} rows")
    print(df_check["sentiment_normalized"].value_counts())

## Embedding Cache Check (Optimization)
If embeddings were computed before, they're saved to Drive to skip the ~30 min encode step in stage 5.

In [ ]:
import numpy as np

# Note: Embedding cache is NOT used by default in the pipeline
# This cell is informational only
EMBED_PATH = f"{OUTPUT_DIR}/embeddings.npy"

if os.path.exists(EMBED_PATH):
    embeddings_cached = np.load(EMBED_PATH)
    print(f"✅ Cached embeddings found: shape {embeddings_cached.shape}")
    print(f"   Note: run_topics.py computes fresh embeddings each run")
    del embeddings_cached
else:
    print("⚠️  No cached embeddings found")
    print(f"   run_topics.py will compute embeddings (~25-35 min on T4)")

## Stage 5 — Topic Modeling / BERTopic (GPU — heaviest stage)
Reads `tweets_with_sentiment.csv`, uses `text_clean_topic` column. Output: `tweets_with_topics.csv`, `topic_info.csv`

In [ ]:
TOPICS_CSV = f"{OUTPUT_DIR}/tweets_with_topics.csv"
TOPIC_INFO = f"{OUTPUT_DIR}/topic_info.csv"

if os.path.exists(TOPICS_CSV) and os.path.exists(TOPIC_INFO):
    df_check = pd.read_csv(TOPICS_CSV)
    ti_check = pd.read_csv(TOPIC_INFO)
    print(f"⏭️  Skipping — tweets_with_topics.csv already exists ({len(df_check):,} rows, {ti_check['Topic'].nunique()} topics)")
else:
    print("Running run_topics.py (GPU — expect 30-60 min if no embedding cache)...")
    t0 = time.time()
    
    result = !RUNTIME_MODE=colab python3 {CODE_DIR}/scripts/run_topics.py 2>&1
    for line in result[-30:]:
        print(line)
    elapsed = time.time() - t0
    
    assert os.path.exists(TOPICS_CSV), f"❌ Expected output not found: {TOPICS_CSV}"
    df_check = pd.read_csv(TOPICS_CSV)
    ti_check = pd.read_csv(TOPIC_INFO)
    outlier_pct = (df_check["topic_id"] == -1).sum() / len(df_check)
    print(f"\n✅ Stage 5 done in {elapsed/60:.1f} min")
    print(f"   Rows: {len(df_check):,}")
    print(f"   Topics found: {ti_check['Topic'].nunique()}")
    print(f"   Outliers (topic=-1): {outlier_pct:.1%}")
    display(ti_check.head(10))

## Stage 6 — Data Validation
Validates pipeline outputs against expected schema and row counts.

In [ ]:
print("Running validate_data_contract.py...")
result = !RUNTIME_MODE=colab python3 {CODE_DIR}/scripts/validate_data_contract.py 2>&1
for line in result:
    print(line)

# Exit if validation fails (mirrors droplet script)
if result and any("FAILED" in line or "ERROR" in line for line in result):
    print("\n❌ Data validation FAILED. Check output above.")
    raise RuntimeError("Pipeline validation failed")
else:
    print("\n✅ Stage 6 complete — all data validated")

## Stage 7 — Generate Manifest
Creates `metadata.json` with run metadata for the dashboard.

In [ ]:
import json
from datetime import datetime

MANIFEST = f"{OUTPUT_DIR}/metadata.json"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

# Calculate total pipeline duration (from notebook start)
if 'notebook_start_time' not in globals():
    notebook_start_time = time.time()
duration = int(time.time() - notebook_start_time)

print(f"Running generate_manifest.py (RUN_ID={RUN_ID}, duration={duration}s)...")
result = !RUNTIME_MODE=colab python3 {CODE_DIR}/scripts/generate_manifest.py {RUN_ID} {duration} 2>&1
for line in result:
    print(line)

assert os.path.exists(MANIFEST), f"❌ Expected output not found: {MANIFEST}"

with open(MANIFEST) as f:
    manifest = json.load(f)
print(f"\n✅ Stage 7 complete — manifest generated")
print(json.dumps(manifest, indent=2))

## Stage 8 — Upload to DO Spaces
Uploads timestamped run to DigitalOcean Spaces for versioning.

In [ ]:
print("Running upload_run.py...")
result = !RUNTIME_MODE=colab python3 {CODE_DIR}/scripts/upload_run.py 2>&1
for line in result:
    print(line)

# Warn if upload fails but don't crash notebook (mirrors droplet script)
if result and any("FAILED" in line or "ERROR" in line for line in result):
    print("\n⚠️  Upload FAILED. Pipeline completed but data not versioned in Spaces.")
else:
    print(f"\n✅ Stage 8 complete — run uploaded to DO Spaces (runs/{RUN_ID}/)")

## Pipeline Summary & Optional Droplet Sync
Pipeline complete! Optionally sync results back to your droplet via rsync.

In [ ]:
# ─── OPTIONAL: uncomment and fill in SSH key path if needed ──────────────────
# SSH_KEY = "/content/drive/MyDrive/keys/your_key.pem"   # path to private key on Drive
# SSH_OPT = f"-e 'ssh -i {SSH_KEY} -p {DROPLET_PORT} -o StrictHostKeyChecking=no'"
# ─────────────────────────────────────────────────────────────────────────────

SSH_OPT = f"-e 'ssh -p {DROPLET_PORT} -o StrictHostKeyChecking=no'"
REMOTE  = f"{DROPLET_USER}@{DROPLET_IP}:/opt/mbg/data/output/"

print(f"Syncing outputs to {REMOTE}...")
sync_cmd = f"rsync -avz --progress {SSH_OPT} {OUTPUT_DIR}/ {REMOTE}"
result = !{sync_cmd} 2>&1
for line in result:
    print(line)

print("\n✅ Sync complete — outputs are on the droplet, ready for the dashboard")